In [2]:
import sympy as sp

In [3]:
from functools import lru_cache

@lru_cache(maxsize = None)
def get_ckn(k: int, n: int, p):
    if k<0 or k>n:
        return sp.Integer(0)
    if n == 0:
        return sp.Integer(1) if k==0 else sp.Integer(0)
    return get_ckn(k-1, n-1, p)/(2*p) + (k+1)*get_ckn(k+1, n-1, p)


In [5]:
x = sp.symbols('x', real = True)
alpha = sp.symbols('alpha', real = True, positive = True)
Ax = sp.symbols('Ax', real = True)

In [7]:
g0 = sp.exp(-alpha*(x-Ax)**2)
g0

exp(-alpha*(-Ax + x)**2)

In [8]:
def get_hermite_gaussian(k):
    return g0.diff(Ax, k)

In [13]:
get_hermite_gaussian(5).simplify()

-8*alpha**3*(Ax - x)*(4*alpha**2*(Ax - x)**4 - 20*alpha*(Ax - x)**2 + 15)*exp(-alpha*(Ax - x)**2)

In [16]:
nmax = 6
for n in range(nmax + 1):
    g = 0
    for k in range(0, n+1):
        g = g + get_ckn(k, n, alpha)*get_hermite_gaussian(k)
    g = g.simplify()
    print(f"g({n}) = {g}")

g(0) = exp(-alpha*(Ax - x)**2)
g(1) = (-Ax + x)*exp(-alpha*(Ax - x)**2)
g(2) = (Ax - x)**2*exp(-alpha*(Ax - x)**2)
g(3) = -Ax*(Ax - x)**2*exp(-alpha*(Ax - x)**2) + x*(Ax - x)**2*exp(-alpha*(Ax - x)**2)
g(4) = (Ax - x)**4*exp(-alpha*(Ax - x)**2)
g(5) = -Ax*(Ax - x)**4*exp(-alpha*(Ax - x)**2) + x*(Ax - x)**4*exp(-alpha*(Ax - x)**2)
g(6) = (Ax - x)**6*exp(-alpha*(Ax - x)**2)


### Implementation of overlap integrals

In [18]:
alpha, beta = sp.symbols('alpha beta', real = True, positive = True)
Ax, Ay, Az = sp.symbols('Ax Ay Az', real = True)
Bx, By, Bz = sp.symbols('Bx By Bz', real = True)

p = alpha + beta
q = alpha*beta/p
RAB2 = (Ax-Bx)**2 + (Ay-By)**2 + (Az-Bz)**2


In [20]:
S00 = (sp.sqrt(sp.pi/p))**3 * sp.exp(-q*RAB2)
S00

pi**(3/2)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(3/2)

In [38]:
def build_derivative_table(base_integral, Lmax, Avars, Bvars):
    Ax, Ay, Az = Avars
    Bx, By, Bz = Bvars
    
    all_idx = [(i, j, L - i - j) for L in range(Lmax+1) 
                               for i in range(L + 1) 
                               for j in range(L+1-i)]


    @lru_cache(maxsize=None)
    def derivative(i,j,k,l,m,n):
        if (i,j,k,l,m,n) == (0,0,0,0,0,0):
            return base_integral
        if i>0:
            return sp.diff(derivative(i-1,j,k,l,m,n),Ax)
        if j>0:
            return sp.diff(derivative(i,j-1,k,l,m,n),Ay)
        if k>0:
            return sp.diff(derivative(i,j,k-1,l,m,n),Az) 
        if l>0:
            return sp.diff(derivative(i,j,k,l-1,m,n),Bx)
        if m>0:
            return sp.diff(derivative(i,j,k,l,m-1,n),By)
        return sp.diff(derivative(i,j,k,l,m,n-1),Bz) 
    
    derivatives_dict = {}
    for (i,j,k) in all_idx:
        for (l,m,n) in all_idx:
            derivatives_dict[(i,j,k,l,m,n)] = derivative(i,j,k,l,m,n)
    return derivatives_dict

In [42]:
Lmax = 3
derivatives_dict = build_derivative_table(S00, Lmax, (Ax,Ay,Az), (Bx,By,Bz))

In [43]:
for key,value in derivatives_dict.items():
    print(key, value)

(0, 0, 0, 0, 0, 0) pi**(3/2)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(3/2)
(0, 0, 0, 0, 0, 1) -pi**(3/2)*alpha*beta*(-2*Az + 2*Bz)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 0, 1, 0) -pi**(3/2)*alpha*beta*(-2*Ay + 2*By)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 1, 0, 0) -pi**(3/2)*alpha*beta*(-2*Ax + 2*Bx)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 0, 0, 2) pi**(3/2)*alpha**2*beta**2*(-2*Az + 2*Bz)**2*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(7/2) - 2*pi**(3/2)*alpha*beta*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2 + (Az - Bz)**2)/(alpha + beta))/(alpha + beta)**(5/2)
(0, 0, 0, 0, 1, 1) pi**(3/2)*alpha**2*beta**2*(-2*Ay + 2*By)*(-2*Az + 2*Bz)*exp(-alpha*beta*((Ax - Bx)**2 + (Ay - By)**2